# Implement Grouped Query Attention from Scratch
### Problem Statement
Grouped Query Attention (GQA), introduced in [Ainslie et al. 2023](https://arxiv.org/abs/2305.13245), is a generalization that sits between Multi-Head Attention (MHA) and Multi-Query Attention (MQA).

- **MHA**: each query head has its own K/V head (`num_kv_heads == num_heads`)
- **MQA**: all query heads share a single K/V head (`num_kv_heads == 1`)
- **GQA**: query heads are divided into groups, each group shares one K/V head (`1 < num_kv_heads < num_heads`)

GQA strikes a balance — it reduces KV cache size (like MQA) while preserving more representational capacity (like MHA). It is used in Llama 2 70B, Llama 3, Mistral, and many modern LLMs.

Your goal is to implement GQA from scratch and validate it against `torch.nn.MultiheadAttention` in the degenerate case where `num_kv_heads == num_heads` (which is identical to MHA).

---

### Requirements

1. **Linear Projections**
   - Q projection: `d_model → d_model` (all query heads)
   - K projection: `d_model → num_kv_heads * d_head` (grouped KV heads)
   - V projection: `d_model → num_kv_heads * d_head` (grouped KV heads)
   - Output projection: `d_model → d_model`

2. **Expand K/V to Match Query Heads**
   - Use `repeat_interleave()` to duplicate each KV group to match the query heads in that group.

3. **Scaled Dot-Product Attention**
   - Compute: `scores = Q @ Kᵀ / sqrt(d_head)`
   - Apply optional mask before softmax.
   - Weight V with attention weights.

4. **Combine Heads and Output**
   - Concatenate all heads and apply final linear projection.

5. **Validate Against MHA**
   - When `num_kv_heads == num_heads`, GQA is identical to MHA.
   - Copy weights and verify outputs match with `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ `num_heads` must be divisible by `num_kv_heads`.
- ✅ Support optional masking.
- ✅ Must match `torch.nn.MultiheadAttention` output when `num_kv_heads == num_heads`.

---

<details>
  <summary>💡 Hint</summary>

  - Q shape after reshape: `(batch, num_heads, seq_len, d_head)`
  - K/V shape after reshape: `(batch, num_kv_heads, seq_len, d_head)`
  - `repeat_factor = num_heads // num_kv_heads`
  - Use `K.repeat_interleave(repeat_factor, dim=1)` to expand K/V from `num_kv_heads` to `num_heads`.
  - When `num_kv_heads == num_heads`, `repeat_factor = 1` so no duplication happens — identical to MHA.
  - When `num_kv_heads == 1`, this becomes MQA.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [ ]:
class GroupedQueryAttention(nn.Module):
    """
    Implements Grouped Query Attention (GQA).
    Query heads are divided into groups, each sharing one K/V head.
    """
    def __init__(self, num_heads, num_kv_heads, d_model):
        super().__init__()
        assert d_model % num_heads == 0
        assert num_heads % num_kv_heads == 0

        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads
        self.repeat_factor = num_heads // num_kv_heads

        # TODO: Create linear projections
        # Q projects to d_model (all query heads)
        # K/V project to num_kv_heads * d_head
        # Output projects d_model -> d_model
        ...

    def forward(self, q, k, v, mask=None):
        """
        Args:
            q (Tensor): Query tensor of shape (batch_size, seq_len, d_model)
            k (Tensor): Key tensor of shape (batch_size, seq_len, d_model)
            v (Tensor): Value tensor of shape (batch_size, seq_len, d_model)
            mask (Tensor, optional): Masking tensor for attention

        Returns:
            Tensor: GQA output of shape (batch_size, seq_len, d_model)
        """
        # TODO: Implement forward pass
        # 1. Project Q, K, V
        # 2. Reshape Q to (batch, num_heads, seq_len, d_head)
        # 3. Reshape K, V to (batch, num_kv_heads, seq_len, d_head)
        # 4. Expand K, V using repeat_interleave to (batch, num_heads, seq_len, d_head)
        # 5. Compute scaled dot-product attention
        # 6. Apply mask if provided
        # 7. Combine heads and apply output projection
        ...

In [ ]:
# Validate: when num_kv_heads == num_heads, GQA == MHA
multihead_attn = torch.nn.MultiheadAttention(
    embed_dim=d_model, num_heads=num_heads, bias=False, batch_first=True
)

custom_attn = GroupedQueryAttention(num_heads, num_kv_heads=num_heads, d_model=d_model)

# Copy weights: when num_kv_heads == num_heads, K/V projections are d_model -> d_model (same as MHA)
custom_attn.Q_w.weight.data = multihead_attn.in_proj_weight[:d_model, :].clone()
custom_attn.K_w.weight.data = multihead_attn.in_proj_weight[d_model:2*d_model, :].clone()
custom_attn.V_w.weight.data = multihead_attn.in_proj_weight[2*d_model:, :].clone()
custom_attn.W_out.weight.data = multihead_attn.out_proj.weight.data.clone()

output_custom = custom_attn(q, k, v)
output_ref, _ = multihead_attn(q, k, v)

print("Custom GQA output:")
print(output_custom)
print("\nPyTorch MHA output:")
print(output_ref)

assert torch.allclose(output_custom, output_ref, atol=1e-06, rtol=1e-05)
print("\n✅ Outputs match! GQA implementation is correct.")

In [ ]:
# Bonus: KV cache comparison across all three variants
d_head = d_model // num_heads
print(f"d_model={d_model}, num_heads={num_heads}, d_head={d_head}")
print(f"")
print(f"MHA (num_kv_heads={num_heads}):  KV cache = {num_heads * d_head * 2} values/token")
print(f"GQA (num_kv_heads=1):            KV cache = {1 * d_head * 2} values/token  (same as MQA)")
print(f"MQA (num_kv_heads=1):            KV cache = {1 * d_head * 2} values/token")
print(f"")
print(f"GQA is a spectrum: num_kv_heads=1 → MQA, num_kv_heads=num_heads → MHA")